In [1]:
from ase.io import write
import subprocess
from pycp2k.templates.GLOBAL.GLOBAL import CP2K
from pycp2k.templates.FORCE_EVAL.xTB_templates import add_xTB_OT
from pycp2k.templates.FORCE_EVAL.PBE_templates import add_PBE_OT
from pycp2k.templates.PRINT.singlepoint import *
from pycp2k.workflows.mk_mace_dataset import load_dataset,get_elements,add_isolated_atoms

In [ ]:
ds=load_dataset("ds.xyz")
symbols_set=get_elements(ds)
ds=add_isolated_atoms(ds,symbols_set)
for system in ds:
    print(system, system.info)

In [ ]:
nfailed=0
for system in ds:
    print(system, system.info)
    calc=CP2K(project_name="mace_xTB",run_type="ENERGY_FORCE") # Need to redefine calculator every time?
    #add_xTB_OT(atoms=system,calc=calc,charge=system.info.get("charge",0),LSD=system.info["oddNumberofElectrons"])
    add_PBE_OT(atoms=system,calc=calc,charge=system.info.get("charge",0),LSD=system.info["oddNumberofElectrons"])
    forces_path=add_print_singlepoint_forces(calc=calc,filename="forces")
    stress_path=add_print_stress_tensor(calc=calc,filename="./")
    try:
        calc.run()
    except Exception as e:
        print(f"Error: {e}")
        output_file=f"{calc.project_name}.out"
        subprocess.run(["tail", "-n", "30", output_file])
        nfailed+=1
        continue
    system.info["E"]=postprocess_energy(calc=calc)
    system.set_array("forces",postprocess_forces(forces_path=forces_path))
    system.info["stress"]=postprocess_stress(stress_path=stress_path,notation="voigt")
    calc.cleanup(quiet=True)

print(f"Number of failed calculations: {nfailed}")
write("ds_ready.xyz",ds,format="extxyz")
    